In [ ]:
import numpy as np
import pandas as pd
import scipy

import matplotlib as mpl
import matplotlib.pyplot as plt

import odrpack as odr
import uncertainties

from uncertainties import nominal_value, std_dev
from uncertainties import ufloat
from uncertainties import unumpy as unp

In [ ]:
HPLANCK = 6.62607015e-34
EMASS = 9.1093837139e-31
ECHARGE = 1.602176634e-19

In [ ]:
df = pd.read_excel("p1ssp.xlsx", sheet_name="Sheet1", decimal = ",")


rawLenArray = np.array(df.iloc[:, 1:5]) * 10**-3

lenMeans = np.mean(rawLenArray, axis = 1)
lenStd = np.std(rawLenArray, axis = 1)

lengths = unp.uarray(lenMeans, lenStd)

rMedias = lengths / 2


V = unp.uarray(df["V (kV)"], len(df["V (kV)"]) * [0.01]) * 1000

lambdas = HPLANCK / unp.sqrt(2*EMASS*ECHARGE*V)

In [ ]:
X = lambdas
Y = rMedias

In [ ]:
#=============================== Blocón de ODR_FIT ===============================

def func(x, beta):
    return beta[0] * x + beta[1]


beta0 = [0.03, 0]

fittingX = unp.nominal_values(X)
fittingY = unp.nominal_values(Y)
fittingXUncertainty = unp.std_devs(X)
fittingYUncertainty = unp.std_devs(Y)

sol = odr.odr_fit(func, fittingX, fittingY, beta0, weight_x=1 / (fittingXUncertainty ** 2),
                  weight_y=1 / (fittingYUncertainty ** 2))

pendiente = ufloat(sol.beta[0], sol.sd_beta[0])  #np.sqrt(sol.cov_beta[0,0]))
ordenada = ufloat(sol.beta[1], sol.sd_beta[1])  #np.sqrt(sol.cov_beta[1,1]))

print("pendiente = {:.4f}".format(pendiente))
print("ordenada = {:.4f}".format(ordenada))

print(f"χ² reducido = {sol.sum_square / (len(unp.nominal_values(X)) - len(sol.beta))}")
#print(f"pearson = {scipy.stats.pearsonr(fittingX, fittingY).statistic}")
print(f"R² = {scipy.stats.pearsonr(fittingX, fittingY).statistic ** 2}")

#=================================================================================

In [ ]:
distancial = 13.5e-2
D1TEO = 213e-12
D2TEO = 123e-12

d1 = distancial / pendiente


print("d1 = {:.4e}".format(d1))
print(f"error relativo = {np.abs((D1TEO-nominal_value(d1))/D1TEO) * 100}")

In [ ]:
fig, ax = plt.subplots(dpi = 250)

ax.errorbar(x = unp.nominal_values(X), y = unp.nominal_values(Y), xerr = unp.std_devs(X), yerr = unp.std_devs(Y), color = "#e9c46a", ls = "", marker = ".", markersize = 2, ecolor = "#e76f51", capsize=2, zorder = 10, elinewidth=.5)

ax.plot(unp.nominal_values(X), func(unp.nominal_values(X), sol.beta), color = "#264653")

ax.set(xlabel = r"$\lambda$ (m)", ylabel = r"$\langle r \rangle$ (m)")

ax.grid(zorder=0,color='gainsboro', linestyle='--', linewidth=0.7)

plt.savefig("ajusted1.png", bbox_inches = "tight")

plt.show()

In [ ]:
df = pd.read_excel("p1ssp.xlsx", sheet_name="Sheet3", decimal = ",")


rawLenArray = np.array(df.iloc[:, 1:5])* 10**-3

lenMeans = np.mean(rawLenArray, axis = 1)
lenStd = np.std(rawLenArray, axis = 1)

lengths = unp.uarray(lenMeans, lenStd)

rMedias = lengths / 2


V = unp.uarray(df["V (kV)"], len(df["V (kV)"]) * [0.01]) * 1000

lambdas = HPLANCK / unp.sqrt(2*EMASS*ECHARGE*V)

In [ ]:
X = lambdas
Y = rMedias

In [ ]:
#=============================== Blocón de ODR_FIT ===============================

def func(x, beta):
    return beta[0] * x + beta[1]


beta0 = [0.03, 0]

fittingX = unp.nominal_values(X)
fittingY = unp.nominal_values(Y)
fittingXUncertainty = unp.std_devs(X)
fittingYUncertainty = unp.std_devs(Y)

sol = odr.odr_fit(func, fittingX, fittingY, beta0, weight_x=1 / (fittingXUncertainty ** 2),
                  weight_y=1 / (fittingYUncertainty ** 2))

pendiente = ufloat(sol.beta[0], sol.sd_beta[0])  #np.sqrt(sol.cov_beta[0,0]))
ordenada = ufloat(sol.beta[1], sol.sd_beta[1])  #np.sqrt(sol.cov_beta[1,1]))

print("pendiente = {:.4f}".format(pendiente))
print("ordenada = {:.4f}".format(ordenada))

print(f"χ² reducido = {sol.sum_square / (len(unp.nominal_values(X)) - len(sol.beta))}")
#print(f"pearson = {scipy.stats.pearsonr(fittingX, fittingY).statistic}")
print(f"R² = {scipy.stats.pearsonr(fittingX, fittingY).statistic ** 2}")

#=================================================================================

In [ ]:
distancial = 13.5e-2
D1TEO = 213e-12
D2TEO = 123e-12

d2 = distancial / pendiente


print("d2 = {:.4e}".format(d2))
print(f"error relativo = {np.abs((D2TEO-nominal_value(d2))/D2TEO) * 100}")

In [ ]:
fig, ax = plt.subplots(dpi = 250)

X = X
Y = Y

mthingie = sol.beta
mthingie[0] = sol.beta[0]
mthingie[1] = sol.beta[1]

ax.errorbar(x = unp.nominal_values(X), y = unp.nominal_values(Y), xerr = unp.std_devs(X), yerr = unp.std_devs(Y), color = "#e9c46a", ls = "", marker = ".", markersize = 2, ecolor = "#e76f51", capsize=2, zorder = 10, elinewidth=.5)

ax.plot(unp.nominal_values(X), func(unp.nominal_values(X), mthingie), color = "#264653")

ax.set(xlabel = r"$\lambda$ (nm)", ylabel = r"$\langle r \rangle$ (m)")

ax.grid(zorder=0,color='gainsboro', linestyle='--', linewidth=0.7)

plt.savefig("ajusted2.png", bbox_inches = "tight")

plt.show()